<center><h1 style="color: #777777;">VIVERO RAÍCES DORADAS - REPORTE OPERATIVO</h1></center>
<center><h2 style="color: #777777;"></h2></center>

<div align="center">
    <table style="width: 60%; float: center;">
        <tr>
            <th style="background-color: #e3d0b3; color: #926e41;">NOMBRE</th>
            <th style="background-color: #e3d0b3; color: #926e41;">TEMPORALIDAD</th>
            <th style="background-color: #e3d0b3; color: #926e41;">OBJETIVO</th>
        </tr>
        <tr>
            <td>Observación de Plantas</td>
            <td>De Octubre a Diciembre 2025</td>
            <td>Informar la capacidad operativa del vivero</td>
        </tr>
    </table>
</div>


<h3 style="color: #777777;"> Paso 1: Seleccionar los datos para el reporte</h3>

In [9]:
import sqlite3
import pandas as pd
from pathlib import Path

def get_observation_data(start_date=None, end_date=None, db_path='../nursery.db'):
    """
    Read observation data from the OBSERVACION_LOTE table with optional date filtering.

    Args:
        start_date(str, optional): Start date in YYYY-MM-DD format. If None, gets all from beginning.
        end_date(str, optional): End date in YYYY-MM-DD format. If None, gets all up to now.
        db_path(str, optional): Path to the SQLite3 database file. Defaults to 'nursery.db

    Returns:
        Pandas.DataFrame: Observation Data
    """

    # Step 1: Check if database exists at the given path
    db_file = Path(db_path)

    # Check if database exists
    if not db_file.exists():
        raise FileNotFoundError(f"Database file not found: {db_file.absolute()}")
    
    try:
        # Step 3: Connect to database 
        conn = sqlite3.connect(db_file)

        # Step 4: Build the query with optional date filtering
        query = """ 
        SELECT * FROM OBSERVACION_LOTE
        WHERE 1=1 --Always true makes it easier to add conditions
        """

        # Add date filters if provided
        params = []
        if start_date:
            query +=  " AND FECHA_OBSERVACION >= ?"
            params.append(start_date)
        if end_date:
            query += " AND FECHA_OBSERVACION <= ?"
            params.append(end_date)

        # Order by date (most recent first)
        query += "ORDER BY FECHA_OBSERVACION DESC"

        # Step 5: Execute the query and get data
        df = pd.read_sql_query(query, conn, params=params if params else None)

        return df
    
    except sqlite3.Error as e:
        print(f"Database error: {e}")
        raise
    finally:
        if 'conn' in locals():
            conn.close()

# Usage in your Jupyter notebook
if __name__ == "__main__":
    try:
        # Example 1: Get all data
        print("=== REPORTANDO LA BASE DE DATOS COMPLETA ===")
        all_data = get_observation_data()

        # Basic analysis
        print(f"\n=== Estadísticas Básicas")
        print(f"Conteo Total de Observaciones: {len(all_data)}")
        print(f"Rango de Fechas: {all_data['FECHA_OBSERVACION'].min()} a {all_data['FECHA_OBSERVACION'].max()}")
        print(f"Cantidad de Lotes Existentes {all_data['ESPECIE_LOTE_ID'].nunique()}")

        # Show column info
        print(f"\n=== Estructura de Datos ===")
        print(f"Columnas disponibles ({len(all_data.columns)}):")
        for col in all_data.columns:
            print(f"  - {col}")

    except Exception as e:
        print(f"Error: {e}")

=== REPORTANDO LA BASE DE DATOS COMPLETA ===

=== Estadísticas Básicas
Conteo Total de Observaciones: 1020
Rango de Fechas: 2025-09-22 a 2025-12-16
Cantidad de Lotes Existentes 35

=== Estructura de Datos ===
Columnas disponibles (8):
  - ID
  - ESPECIE_LOTE_ID
  - FECHA_OBSERVACION
  - CANTIDAD_PLANTAS_VIVAS
  - ALTURA_PROMEDIO_CM
  - ETAPA_CRECIMIENTO
  - INDICADOR_SALUD
  - NOTAS
